# Matcher: Export to OpenVINO and Run on CPU

This notebook demonstrates the full workflow:

1. **Fit** a Matcher model on a reference image + mask
2. **Export** the fitted model to OpenVINO IR format
3. **Run inference** with the OpenVINO model on CPU
4. **Visualize** the results side-by-side with PyTorch predictions

In [ ]:
# Copyright (C) 2026 Intel Corporation
# SPDX-License-Identifier: Apache-2.0

from __future__ import annotations

from pathlib import Path
from time import time

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F

from instantlearn.data import Sample
from instantlearn.data.utils.image import read_image
from instantlearn.models import Matcher
from instantlearn.utils.constants import Backend
from instantlearn.visualizer import render_predictions

In [ ]:
EXAMPLES_DIR = Path("assets/coco")
EXPORT_DIR = Path("output/matcher_openvino")

REF_IMAGE = EXAMPLES_DIR / "000000286874.jpg"
REF_MASK = EXAMPLES_DIR / "000000286874_mask.png"
TARGET_IMAGES = [
    EXAMPLES_DIR / "000000390341.jpg",
    EXAMPLES_DIR / "000000173279.jpg",
    EXAMPLES_DIR / "000000267704.jpg",
]

TORCH_DEVICE = "cuda"  # Change to xpu or cpu
OV_DEVICE = "CPU"

## 1. Fit the Matcher model

We use a lightweight configuration (`dinov3_small` + `SAM-HQ-base`) so the
example runs quickly on CPU.

In [ ]:
model = Matcher(
    device=TORCH_DEVICE,
    encoder_model="dinov3_small",
    sam="SAM-HQ-base",
    confidence_threshold=0.25,  # Low confidence to show more predictions for visualization purposes. Adjust as needed.
)

ref_sample = Sample(
    image_path=str(REF_IMAGE),
    mask_paths=str(REF_MASK),
)

model.fit(ref_sample)
print("Matcher fitted on reference image.")

## 2. PyTorch baseline predictions

Run the model in PyTorch to have a baseline for comparison.

In [ ]:
target_samples = [Sample(image_path=str(p)) for p in TARGET_IMAGES]
pytorch_preds = model.predict(target_samples)

for i, pred in enumerate(pytorch_preds):
    n_masks = pred["pred_masks"].shape[0]
    scores = pred["pred_scores"]
    print(f"  Target {i}: {n_masks} masks, scores {scores.tolist()}")

## 3. Export to OpenVINO

The export pipeline: PyTorch &rarr; ONNX &rarr; OpenVINO IR (`.xml` + `.bin`).

Reference features from `fit()` are frozen into the graph, so the exported
model takes only a target image as input.

In [ ]:
ov_path = model.export(
    export_dir=EXPORT_DIR,
    backend=Backend.OPENVINO,
    compress_to_fp16=False,
)

print(f"Exported to: {ov_path}")
print(f"Files: {[f.name for f in EXPORT_DIR.iterdir()]}")

## 4. Load and run the OpenVINO model on CPU

In [ ]:
import openvino

core = openvino.Core()
ov_model = core.read_model(str(ov_path))
compiled = core.compile_model(ov_model, OV_DEVICE)

input_shape = compiled.input(0).shape  # [1, 3, H, W]
print(f"Model input shape: {input_shape}")
print(f"Outputs: {[o.get_any_name() for o in compiled.outputs]}")

In [ ]:
def predict_openvino(
    compiled_model: openvino.CompiledModel,
    image_path: Path,
) -> dict[str, np.ndarray]:
    """Run a single image through the compiled OpenVINO model."""
    image = read_image(str(image_path))  # [3, H, W] float32 torch tensor
    tensor = image.unsqueeze(0)  # [1, 3, H, W]

    # Resize to the model's static input shape
    expected_h, expected_w = compiled_model.input(0).shape[2:]
    tensor = F.interpolate(tensor, size=(expected_h, expected_w), mode="bilinear", align_corners=False)

    result = compiled_model(tensor.numpy())
    masks, scores, labels = result.values()
    return {"masks": np.array(masks), "scores": np.array(scores), "labels": np.array(labels)}

In [ ]:
ov_results = []
for target_image in TARGET_IMAGES:
    start = time()
    result = predict_openvino(compiled, target_image)
    end = time()
    ov_results.append(result)
    print(f"Inference on {target_image.name} took {end - start:.3f} seconds")

for i, res in enumerate(ov_results):
    n = (res["scores"] > 0).sum()
    print(f"  Target {i}: {n} active masks (scores > 0), scores {res['scores'].tolist()}")

## 5. Visualize: PyTorch vs OpenVINO

Overlay predictions from both backends on the target images.

In [ ]:
target_images = [read_image(str(p)) for p in TARGET_IMAGES]

fig, axes = plt.subplots(len(TARGET_IMAGES), 2, figsize=(14, 7 * len(TARGET_IMAGES)))

for row, (img, pt_pred, ov_res) in enumerate(zip(target_images, pytorch_preds, ov_results, strict=False)):
    img_hwc = img.permute(1, 2, 0).numpy()

    # PyTorch predictions (left)
    n_pt = pt_pred["pred_masks"].shape[0]
    pt_vis = {**pt_pred, "pred_labels": torch.arange(n_pt)}
    cmap = {i: [255, 80, 80] for i in range(n_pt)}
    rendered_pt = render_predictions(img_hwc, pt_vis, cmap)
    axes[row, 0].imshow(rendered_pt)
    axes[row, 0].set_title(f"PyTorch — {n_pt} masks", fontsize=13)
    axes[row, 0].axis("off")

    # OpenVINO predictions (right)
    ov_masks = torch.from_numpy(ov_res["masks"])
    ov_scores = torch.from_numpy(ov_res["scores"])
    ov_labels = torch.from_numpy(ov_res["labels"])

    # Filter to active masks (score > 0)
    keep = ov_scores > 0
    ov_masks = ov_masks[keep]
    ov_scores = ov_scores[keep]
    ov_labels = ov_labels[keep]

    # Resize masks back to original image dimensions
    if ov_masks.numel() > 0 and ov_masks.shape[-2:] != img.shape[-2:]:
        ov_masks = F.interpolate(
            ov_masks.unsqueeze(0).float(),
            size=img.shape[-2:],
            mode="nearest",
        ).squeeze(0)
    ov_masks = (ov_masks > 0.5).to(torch.uint8)

    n_ov = ov_masks.shape[0]
    ov_vis = {
        "pred_masks": ov_masks,
        "pred_scores": ov_scores,
        "pred_labels": torch.arange(n_ov),
    }
    cmap_ov = {i: [80, 180, 255] for i in range(n_ov)}
    rendered_ov = render_predictions(img_hwc, ov_vis, cmap_ov)
    axes[row, 1].imshow(rendered_ov)
    axes[row, 1].set_title(f"OpenVINO CPU — {n_ov} masks", fontsize=13)
    axes[row, 1].axis("off")

fig.suptitle("Matcher: PyTorch vs OpenVINO (CPU)", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()

## Notes

- The exported model has **static input shape** matching the encoder's input size
  (e.g., 512×512). Target images are resized before inference.
- Reference features from `fit()` are **baked into the graph** — the exported model
  takes only `target_image [1, 3, H, W]` as input.
- The export graph produces **one mask per category** (semantic segmentation).
  For instance-level separation, apply connected-component analysis or the
  `PostProcessorPipeline` on the host side.
- Set `compress_to_fp16=True` in `export()` to halve the model size with
  minimal accuracy impact.